# Capítulo 6 — Defesas: treino adversarial, compressão e detecção

## Missão
Hora de proteger o classificador de dígitos que você atacou no cap. 1. Os capítulos 8 e 9 do livro listam as famílias de defesa. Você vai implementar e medir quatro:
1. **treino adversarial**: treinar com exemplos PGD;
2. **compressão das entradas** (*feature squeezing*): quantizar os pixels;
3. **detecção** de exemplos adversariais pela diferença entre a previsão normal e a comprimida;
4. **saneamento** do treino contra o envenenamento (remoção de pontos suspeitos).

E vai aprender a regra de ouro: avalie sempre contra um atacante que **conhece a defesa**.

**No fim deste caderno você terá:** o módulo `meu_escudo/defesas.py` (treino adversarial, compressão, detector por compressão e saneamento por vizinhos) e a tabela de robustez antes e depois.

**Leitura essencial (opcional, livro em inglês):**
- 8.2 Security Threats · PDF p. 135
- 8.4 Poisoned Data Defense · PDF p. 137
- 8.5 Mixup Inference Against Adversarial Attacks · PDF p. 138
- 9.2 Categories of Defenses · PDF p. 145
- 9.4 Discussions and Open Issues · PDF p. 150

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap06_defesas_treino_adversarial_compressao_e_detec.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_escudo/
os.makedirs("../meu_escudo", exist_ok=True)
open("../meu_escudo/__init__.py", "a").close()        # meu_escudo/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_escudo")

## Ideia em 30 segundos
- **Treino adversarial:** em vez de minimizar o erro nos dados, minimize o erro no **pior** exemplo dentro da bola ε (min-max). Na prática: gere PGD contra o modelo atual, junte ao treino e repita.
- **Compressão:** arredondar os pixels (por exemplo, para 0 ou 1) apaga perturbações pequenas.
- **Detecção:** se a previsão muda muito depois de comprimir, desconfie.

Defesas que só **escondem o gradiente** caem diante de um atacante adaptativo. Por isso a avaliação honesta ataca o sistema inteiro.

### 6.1 **As defesas.** Crie `defesas.py` com:
- `treino_adversarial(X, y, eps, rodadas=4)`: começa com uma `LogisticRegression` normal; em cada rodada, gera PGD (do seu `ataques.py`) contra o modelo atual e retreina com os dados originais + os adversariais. Devolve o modelo;
- `comprimir(X, niveis=2)`: quantiza para `niveis` valores entre 0 e 1;
- `escore_deteccao(modelo, X, niveis=2)`: a distância L1 entre as probabilidades em `X` e em `comprimir(X)`;
- `sanear(X, y, k=10, limiar=0.5)`: remove os exemplos cujo rótulo discorda de mais de `limiar` dos seus k vizinhos.

_A célula grava `meu_escudo/defesas.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_escudo/defesas.py


### 6.2 **Treino adversarial.** Treine três modelos:
- o normal (`SoftmaxLinear(eps=0)`);
- o adversarial "ingênuo": uma `LogisticRegression` retreinada uma vez com os exemplos PGD do modelo normal;
- o adversarial de verdade (`treino_adversarial` com ε = 0,1).

Meça cada um em dados limpos e sob PGD com ε = 0,05, 0,1 e 0,2, sempre atacando o **próprio** modelo.

### 6.3 **Compressão como pré-processamento.** Compare, no modelo normal, a acurácia sob PGD (ε = 0,1 e 0,2) **com e sem** comprimir a entrada para 2 níveis antes de prever. Depois responda ao atacante adaptativo: ele sabe que você comprime e usa ε = 0,5, o suficiente para mudar o arredondamento. A defesa resiste?

### 6.4 **Detector.** Calcule o `escore_deteccao` em 500 exemplos limpos e nos mesmos 500 atacados com PGD (ε = 0,1). Desenhe os dois histogramas e calcule a AUC do detector. Escolha o limiar com 5% de falsos alarmes e diga quantos ataques ele pega.

### 6.5 **Saneamento contra o envenenamento.** Inverta 15% dos rótulos do treino (`inverter_aleatorio` do cap. 2), rode o `sanear` e mostre:
- quantos exemplos foram removidos e quantos deles estavam de fato envenenados (a precisão da limpeza);
- a acurácia de uma `LogisticRegression` treinada sem saneamento e com saneamento.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
